# Tamil + Telugu Cramér's V Relationship Analysis

This notebook finds pairwise Cramér's V relationships and high-confidence conditional relationships for **Sentiment, Sarcasm, Vulgarity, Abuse and Target**.

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import chi2_contingency
from pathlib import Path

TAMIL_FILE = 'tamil_train.csv'
TELUGU_FILE = 'telugu_train.csv'
OUTPUT_DIR = Path('relationship_analysis')
OUTPUT_DIR.mkdir(exist_ok=True)
LABEL_COLUMNS = ['sentiment','sarcasm','vulgar','abuse','target']
MIN_CONDITION_COUNT = 5
CONDITIONAL_THRESHOLD = 0.95

In [ ]:
def load_labels(file_path):
    df = pd.read_csv(file_path)
    cols = [c for c in LABEL_COLUMNS if c in df.columns]
    df = df[cols].copy()
    for c in cols:
        df[c] = df[c].astype(str).str.strip().str.lower()
    return df

def cramers_v(a, b):
    table = pd.crosstab(a, b)
    chi2, p, _, _ = chi2_contingency(table)
    n = table.to_numpy().sum()
    r, c = table.shape
    den = n * min(r - 1, c - 1)
    return (np.sqrt(chi2 / den) if den else 0.0), p

def pairwise_cramers(df, name):
    rows = []
    for a, b in combinations(df.columns, 2):
        v, p = cramers_v(df[a], df[b])
        rows.append({'dataset':name,'relationship':f'{a} <-> {b}','cramers_v':v,'p_value':p})
    return pd.DataFrame(rows).sort_values('cramers_v', ascending=False)

## Conditional relationships

For a condition A and outcome B:

$$P(B=b|A=a)=\\frac{count(A=a,B=b)}{count(A=a)}$$

Only conditions supported by at least `MIN_CONDITION_COUNT` samples and having probability >= 95% are reported.

In [ ]:
def conditional_relationships(df, name, n_conditions):
    rows = []
    for condition_cols in combinations(df.columns, n_conditions):
        remaining = [c for c in df.columns if c not in condition_cols]
        for outcome in remaining:
            group_cols = list(condition_cols) + [outcome]
            counts = df.groupby(group_cols).size().reset_index(name='outcome_count')
            totals = df.groupby(list(condition_cols)).size().reset_index(name='condition_count')
            counts = counts.merge(totals, on=list(condition_cols))
            counts['percentage'] = counts['outcome_count'] / counts['condition_count'] * 100
            for _, row in counts.iterrows():
                if row['condition_count'] < MIN_CONDITION_COUNT or row['percentage'] < CONDITIONAL_THRESHOLD * 100:
                    continue
                condition = ' + '.join(f'{c}={row[c]}' for c in condition_cols)
                rows.append({
                    'dataset':name,
                    'type':f'{n_conditions + 1}-way',
                    'condition':condition,
                    'outcome':f'{outcome}={row[outcome]}',
                    'condition_count':int(row['condition_count']),
                    'outcome_count':int(row['outcome_count']),
                    'percentage':row['percentage']
                })
    if not rows:
        return pd.DataFrame(columns=['dataset','type','condition','outcome','condition_count','outcome_count','percentage'])
    return pd.DataFrame(rows).sort_values(['percentage','condition_count'], ascending=[False,False])

In [ ]:
def analyze(file_path, name):
    df = load_labels(file_path)
    print(f'\\n===== {name.upper()} =====')
    print('Rows:', len(df))
    print('Labels:', list(df.columns))

    cramers = pairwise_cramers(df, name)
    pairs = conditional_relationships(df, name, 1)
    triplets = conditional_relationships(df, name, 2)
    four_way = conditional_relationships(df, name, 3)

    cramers.to_csv(OUTPUT_DIR / f'{name.lower()}_cramers_v.csv', index=False)
    pairs.to_csv(OUTPUT_DIR / f'{name.lower()}_conditional_pairs_gt95.csv', index=False)
    triplets.to_csv(OUTPUT_DIR / f'{name.lower()}_conditional_triplets_gt95.csv', index=False)
    four_way.to_csv(OUTPUT_DIR / f'{name.lower()}_conditional_four_way_gt95.csv', index=False)

    all_rel = pd.concat([pairs, triplets, four_way], ignore_index=True)
    all_rel.to_csv(OUTPUT_DIR / f'{name.lower()}_all_gt95_relationships.csv', index=False)

    print('\\nCramér\'s V')
    display(cramers)
    print('\\nAll conditional relationships >= 95%')
    display(all_rel)
    return df, cramers, pairs, triplets, four_way, all_rel

In [ ]:
tamil = analyze(TAMIL_FILE, 'Tamil')
telugu = analyze(TELUGU_FILE, 'Telugu')

In [ ]:
comparison = tamil[1][['relationship','cramers_v']].rename(columns={'cramers_v':'tamil_v'})
comparison = comparison.merge(
    telugu[1][['relationship','cramers_v']].rename(columns={'cramers_v':'telugu_v'}),
    on='relationship', how='outer'
)
comparison['difference'] = comparison['telugu_v'] - comparison['tamil_v']
comparison = comparison.sort_values('telugu_v', ascending=False)
display(comparison)
comparison.to_csv(OUTPUT_DIR / 'tamil_vs_telugu_cramers_v.csv', index=False)

## Interpretation

Cramér's V measures association; it is not itself a prediction rule. Conditional percentages must be interpreted in the correct direction. Always inspect `condition_count` together with the percentage because a 100% relationship based on a very small sample is less reliable than one based on hundreds of samples.